# Overview & Navigation – your project intro + a Colab badge that only shows on GitHub (hidden when opened in Colab).


**Project Intro**  
Zero‑shot job classification that references **MNPS Roles** and **MNPS KSACs**. After the first pass, results are **refined** by comparing the model’s justifications to **Ground Truth Masterfile.csv** justifications (post‑classification few‑shot via similarity). No few‑shot exemplars are embedded in the prompt itself.

**Minor Sub‑Group Policy**  
Allowed: **Lead, I, II, III**. If the model implies **IV/4**, interpret as **Lead** only when KSACs/Functions show leadership/escalation/mentorship; otherwise map to **III**.

**Open in Colab (visible only on GitHub):**  

<div id="github-badge">
<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/Main-Version-2/MNPS_ZeroShot_GT_Refine_FINAL.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>
</div>

<!-- Hide badge when opened in Colab -->
<script>
try {
  if (typeof google !== 'undefined' && google.colab) {
    var el = document.getElementById('github-badge');
    if (el) el.style.display = 'none';
  }
} catch (e) {}
</script>

# Environment Setup – pip installs + imports (so yes, you do still have a dedicated setup section).

In [ ]:
!pip -q install scikit-learn openai pandas

import os, re, json, time, datetime, shutil
from typing import Dict, List, Tuple
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# Inputs & Configuration – file paths, model settings, constants.

In [ ]:
# Input files
BATCH_INPUT_CSV  = "Batch Input.csv"
ROLES_CSV        = "MNPS Roles.csv"
KSACS_CSV        = "MNPS KSACs.csv"
GT_CSV           = "Ground Truth Masterfile.csv"
COMP_EXT_CSV     = "Competency Extended Descriptions.csv"    # optional
LOMINGER_CSV     = "Korn_Ferry Lominger 38 Competencies.csv" # optional

# Outputs
OUTPUT_PRED_CSV  = "classified_job_descriptions_pre_refine.csv"
OUTPUT_FINAL_CSV = "classified_job_descriptions_final.csv"
OUTPUT_CANONICAL = "classified_job_descriptions.csv"  # canonical final

# LLM
MODEL_PROVIDER   = "openai"
MODEL_NAME       = "gpt-4o-mini"
TEMPERATURE      = 0.2
TOP_K_CANDIDATES = 10


# Zero‑Shot Prompt – your Section 4 text verbatim, with the minor-level policy note.


```
Objective: Evaluate and group jobs from the "Batch Input.csv" file based on similarities in job functions, not job titles.

Process:

- Compare all jobs against each other using the attributes listed in the file: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary.
- Compare each job with reference sources using the same attributes. I have attached the reference sources for you, including "MNPS Roles.csv" and "MNPS KSACs.csv".
- Group jobs based on similarities into:
  - Major role groupings from the comprehensive list provided in "MNPS Roles.csv" (e.g., Specialist, Analyst, Manager, Technician, Para Pro, Advisor, etc.)
  - Minor sub-groupings (e.g., Specialist I, II, III, IV)
- **Crucially, use the "MNPS Roles.csv" and "MNPS KSACs.csv" documents as the definitive and comprehensive lists of valid Major Role Groups and their corresponding Knowledge, Skills, Abilities, and Competencies (KSACs) to guide your classification.**
- Use the remaining documents ("Competency Extended Descriptions.csv" and "Korn_Ferry Lominger 38 Competencies.csv") to help you clarify subtle differences in role groupings and sub-groupings and to inform the grouping justification.
- Use a more qualitative, holistic assessment focused on functional alignment with KSACs rather than a quantitative scoring approach with defined complexity metrics

Output Format:

- Create a table with the following columns:
  - Original Job Title
  - New Job Title
  - Major Role Group
  - Minor Sub-Group
  - Justification for Grouping

- Provide an accompanying narrative explaining the rationale behind the groupings and any notable patterns or insights discovered during the analysis.

Job Title Convention:

- Follow the format: "[Function] [Role] [Level]" (e.g., "Collections Specialist II", "Accounts Payable Specialist III"). Ensure the Role comes from the list in "MNPS Roles.csv".

Additional Guidelines:

- Ensure all sources used are cited properly in the justification.
- Focus on the nature of the work performed rather than just the job titles.
- Consider the complexity of tasks, level of responsibility, and required competencies when determining groupings.
- Provide clear explanations for why each job was classified as it was, referencing specific job attributes and external benchmarks, and explicitly referencing the relevant KSACs and roles from the provided documents.
- Run in batch over "Batch Input.csv" with conservative settings (temperature=0.2) for reproducibility.
```

**Minor Sub‑Group Policy Note:** Approved minor levels are **Lead, I, II, III**. Interpret any model‑proposed **IV/4** as **Lead** *if* the KSACs/Functions clearly indicate leadership/escalation/mentorship; otherwise map to **III**.

# Load Reference Data & Build Closed Role Set – reads MNPS Roles, MNPS KSACs, and Ground Truth Masterfile.csv; constructs the closed label set and the KSAC corpus.

In [ ]:
def read_csv_robust(path: str):
    for enc in ["utf-8", "latin1", "windows-1252"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    return pd.read_csv(path, encoding="latin1", errors="ignore")

def norm_col(s: str) -> str:
    return re.sub(r'[^a-z0-9]+','_',str(s).strip().lower())

roles_df = read_csv_robust(ROLES_CSV)
ksacs_df = read_csv_robust(KSACS_CSV)
gt_df    = read_csv_robust(GT_CSV)

roles_df.columns = [norm_col(c) for c in roles_df.columns]
ksacs_df.columns = [norm_col(c) for c in ksacs_df.columns]
gt_df.columns    = [norm_col(c) for c in gt_df.columns]

role_col = [c for c in roles_df.columns if "role" in c][-1]
ks_role  = [c for c in ksacs_df.columns if "role" in c][-1]
ks_text  = [c for c in ksacs_df.columns if "ksac" in c or "knowledge" in c][-1]

# Closed role set from MNPS Roles.csv
VALID_ROLES = set(roles_df[role_col].astype(str).str.strip())

# Role -> KSAC text corpus from MNPS KSACs.csv
role_ksac = (
    ksacs_df.groupby(ks_role)[ks_text]
            .apply(lambda s: " ".join(map(lambda x: "" if pd.isna(x) else str(x), s)))
            .to_dict()
)

# Detect GT columns for refinement/eval
gt_name = [c for c in gt_df.columns if "job" in c and "name" in c]
gt_major = [c for c in gt_df.columns if "major" in c and "role" in c]
gt_minor = [c for c in gt_df.columns if "minor" in c]
gt_just  = [c for c in gt_df.columns if "justif" in c]

GT_COLS = {
    "name":  gt_name[0]  if gt_name  else None,
    "major": gt_major[0] if gt_major else None,
    "minor": gt_minor[0] if gt_minor else None,
    "just":  gt_just[0]  if gt_just  else None,
}

print("Closed role set size:", len(VALID_ROLES))


# Classification Utilities – normalizers (Lead/I/II/III + IV→Lead/III rule), TF‑IDF retrieval, prompt builder, and LLM JSON call.

In [ ]:
APPROVED_MINOR = {"Lead","I","II","III"}

def looks_like_lead(fields: Dict[str,str]) -> bool:
    text = " ".join([
        str(fields.get("Essential Functions","")),
        str(fields.get("Knowledge, Skills and Abilities","")),
        str(fields.get("Position Summary",""))
    ]).lower()
    lead_signals = [
        "lead", "leads", "team lead", "mentors", "mentorship",
        "coaches others", "escalation", "tier 3", "senior-most",
        "principal", "subject matter expert", "sme",
        "oversees", "coordinates others", "assigns work", "guides staff"
    ]
    return any(sig in text for sig in lead_signals)

def normalize_minor(sub: str, fields: Dict[str,str]) -> str:
    s = "" if sub is None else str(sub).strip()
    if s.lower() == "lead":
        return "Lead"
    if s in {"I","II","III"}:
        return s
    m = re.search(r"\b(I|II|III|IV)\b", s.upper())
    if m:
        roman = m.group(1)
        if roman in {"I","II","III"}:
            return roman
        if roman == "IV":
            return "Lead" if looks_like_lead(fields) else "III"
    if s.isdigit():
        if s in {"1","2","3"}:
            return {"1":"I","2":"II","3":"III"}[s]
        if s == "4":
            return "Lead" if looks_like_lead(fields) else "III"
    m2 = re.search(r"\b(level|tier)\s*(1|2|3|4|i|ii|iii|iv)\b", s.lower())
    if m2:
        val = m2.group(2).upper()
        if val in {"I","II","III"}:
            return val
        if val in {"IV","4"}:
            return "Lead" if looks_like_lead(fields) else "III"
    return "Lead" if looks_like_lead(fields) else "III"

# TF-IDF retrieval over role KSAC corpus
role_names = list(role_ksac.keys())
role_texts = [role_ksac.get(r, "") for r in role_names]
vec = TfidfVectorizer(min_df=1, max_df=0.9, ngram_range=(1,2))
X_roles = vec.fit_transform(role_texts)

def topK_roles(desc_blob: str, K: int = 10):
    q = vec.transform([desc_blob])
    sims = cosine_similarity(q, X_roles).ravel()
    idx = sims.argsort()[::-1][:K]
    return [role_names[i] for i in idx]

PROMPT_TMPL = """    Objective: Evaluate and group this MNPS job based on similarities in job functions, not job titles.
Use MNPS Roles (closed label set) and KSACs. Choose only from the provided candidate roles.
Approved minor sub-groups: "Lead", "I", "II", "III". If "IV/4" is inferred, interpret as "Lead" only if KSACs/Functions show leadership; otherwise "III".
Output STRICT JSON with keys: major_role_group, minor_sub_group, new_job_title, grouping_justification.
Cite sources in justification (e.g., "MNPS KSACs: Technician", "MNPS Roles").
Candidate Roles: {candidates}
Position Summary: {summary}
Essential Functions: {functions}
Education: {education}
Work Experience: {experience}
Licenses/Certifications: {licenses}
Knowledge, Skills and Abilities: {ksac}
"""

def build_prompt(row: pd.Series):
    fields = {
        "Position Summary": row.get("Position Summary",""),
        "Essential Functions": row.get("Essential Functions",""),
        "Education": row.get("Education",""),
        "Work Experience": row.get("Work Experience",""),
        "Licenses/Certifications": row.get("Licenses and Certifications",""),
        "Knowledge, Skills and Abilities": row.get("Knowledge, Skills and Abilities",""),
    }
    blob = " ".join(str(v) for v in fields.values())
    candidates = topK_roles(blob, K=TOP_K_CANDIDATES)
    prompt = PROMPT_TMPL.format(
        candidates=", ".join(candidates),
        summary=fields["Position Summary"][:2000],
        functions=fields["Essential Functions"][:2000],
        education=fields["Education"][:1200],
        experience=fields["Work Experience"][:1200],
        licenses=fields["Licenses/Certifications"][:800],
        ksac=fields["Knowledge, Skills and Abilities"][:1200],
    )
    return prompt, fields

def call_llm_json(prompt: str) -> str:
    if MODEL_PROVIDER == "openai":
        from openai import OpenAI
        if not os.getenv("OPENAI_API_KEY"):
            raise RuntimeError("OPENAI_API_KEY not set in environment.")
        client = OpenAI()
        resp = client.chat.completions.create(
            model=MODEL_NAME,
            temperature=TEMPERATURE,
            response_format={"type": "json_object"},
            messages=[
                {"role":"system","content":"You are a careful classifier that returns strict JSON."},
                {"role":"user",  "content": prompt},
            ]
        )
        return resp.choices[0].message.content
    else:
        raise NotImplementedError("MODEL_PROVIDER not supported here.")


# Zero‑Shot Classification + GT Refinement & Evaluation – first pass classification (no few-shot in prompt), then justification-similarity refinement against Ground Truth and optional eval on overlapping titles.

In [ ]:
# Load batch input
def read_csv_robust(path: str):
    for enc in ["utf-8", "latin1", "windows-1252"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    return pd.read_csv(path, encoding="latin1", errors="ignore")

in_df = read_csv_robust(BATCH_INPUT_CSV)
required = [
    "Job Description Name","Position Summary","Education","Work Experience",
    "Essential Functions","Licenses and Certifications","Knowledge, Skills and Abilities"
]
miss = [c for c in required if c not in in_df.columns]
if miss:
    raise ValueError(f"Missing required columns in Batch Input: {miss}")

work_df = in_df.reset_index().rename(columns={"index":"row_id"}).copy()

# First pass predictions
preds, logs = [], []
for _, row in work_df.iterrows():
    prompt, fields = build_prompt(row)
    raw = call_llm_json(prompt)
    rec = json.loads(raw)
    role = (rec.get("major_role_group","") or "").strip()
    if role not in VALID_ROLES:
        raise ValueError(f"Predicted role '{role}' not in MNPS Roles.csv")
    minor = normalize_minor(rec.get("minor_sub_group",""), fields)
    rec["minor_sub_group"] = minor
    logs.append({"row_id": row["row_id"], "job_name": row["Job Description Name"],
                 "prompt_preview": prompt[:1000], "raw_response": raw})
    preds.append(rec)

pred_df = pd.DataFrame(preds)
out_pre = pd.concat([work_df[["row_id","Job Description Name"]], pred_df], axis=1)
out_pre["original_job_title"] = out_pre["Job Description Name"]
out_pre = out_pre[[
    "original_job_title","new_job_title","major_role_group","minor_sub_group","grouping_justification",
    "Job Description Name","row_id"
]]
out_pre.to_csv(OUTPUT_PRED_CSV, index=False, encoding="utf-8")
pd.DataFrame(logs).to_csv("classification_decision_log.csv", index=False, encoding="utf-8")
print("Saved pre-refinement predictions ->", OUTPUT_PRED_CSV)

# Build GT similarity space (justifications + KSACs)
vec_gt = TfidfVectorizer(min_df=1, max_df=0.95, ngram_range=(1,2))
gt_join = gt_df.copy()
text_cols = []
gt_just_col = [c for c in gt_join.columns if "justif" in c]
if gt_just_col:
    text_cols.append(gt_just_col[0])
ks_map = {k: role_ksac.get(k,"") for k in VALID_ROLES}
gt_major_col = [c for c in gt_join.columns if "major" in c and "role" in c]
if gt_major_col:
    gt_join["__ksac_blob__"] = gt_join[gt_major_col[0]].map(ks_map)
    text_cols.append("__ksac_blob__")
gt_join["__text__"] = gt_join[text_cols].apply(lambda r: " ".join(map(str, r.values)), axis=1)
X_gt = vec_gt.fit_transform(gt_join["__text__"])

from sklearn.metrics.pairwise import cosine_similarity as cos

enrich = work_df.merge(out_pre[["row_id","major_role_group","minor_sub_group","new_job_title","grouping_justification"]],
                       on="row_id", how="left")

finals, logs_refine = [], []
for _, r in enrich.iterrows():
    q_text = " ".join([
        str(r.get("grouping_justification","")),
        str(r.get("Position Summary","")),
        str(r.get("Essential Functions","")),
        str(r.get("Knowledge, Skills and Abilities",""))
    ])
    q = vec_gt.transform([q_text])
    sims = cos(q, X_gt).ravel()
    top = sims.argsort()[::-1][:5]

    major_pred = r["major_role_group"]
    minor_pred = r["minor_sub_group"]
    title_pred = r["new_job_title"]
    just_pred  = r["grouping_justification"]
    reason = []

    best_idx, best_sim = top[0], sims[top[0]]
    gt_role = gt_join.iloc[best_idx][gt_major_col[0]] if gt_major_col else ""
    gt_minor_col = [c for c in gt_join.columns if "minor" in c]
    gt_minor_val = gt_join.iloc[best_idx][gt_minor_col[0]] if gt_minor_col else ""
    gt_just_col0 = gt_just_col[0] if gt_just_col else None
    gt_just_val  = gt_join.iloc[best_idx][gt_just_col0] if gt_just_col0 else ""

    if best_sim >= 0.35 and gt_role and gt_role in VALID_ROLES and gt_role != major_pred:
        reason.append(f"Role refined from {major_pred} -> {gt_role} based on GT similarity {best_sim:.2f}")
        major_pred = gt_role

    if gt_minor_val:
        fields = {
            "Position Summary": r.get("Position Summary",""),
            "Essential Functions": r.get("Essential Functions",""),
            "Knowledge, Skills and Abilities": r.get("Knowledge, Skills and Abilities","")
        }
        minor_pred = normalize_minor(gt_minor_val, fields)
        reason.append(f"Minor set to {minor_pred} using GT exemplar")

    final_just = (str(just_pred) + " | Refined using GT exemplar: " + str(gt_just_val)).strip()

    finals.append({"row_id": r["row_id"], "final_major_role_group": major_pred,
                   "final_minor_sub_group": minor_pred, "final_new_job_title": title_pred,
                   "final_grouping_justification": final_just})
    logs_refine.append({"row_id": r["row_id"], "job_name": r["Job Description Name"],
                        "refine_reason": "; ".join(reason)})

final_df = pd.DataFrame(finals)
out_final = (work_df[["row_id","Job Description Name"]]
             .merge(final_df, on="row_id", how="left"))
out_final["original_job_title"] = out_final["Job Description Name"]
out_final = out_final[[
    "original_job_title","final_new_job_title","final_major_role_group","final_minor_sub_group","final_grouping_justification",
    "Job Description Name","row_id"
]].rename(columns={
    "final_new_job_title":"new_job_title",
    "final_major_role_group":"major_role_group",
    "final_minor_sub_group":"minor_sub_group",
    "final_grouping_justification":"grouping_justification"
})

# Optional: Simple evaluation if GT has overlapping names
if GT_COLS["name"] and GT_COLS["major"] and GT_COLS["minor"]:
    gt_eval = gt_df.copy()
    gt_eval["_join_name"] = gt_eval[GT_COLS["name"]].astype(str).str.strip().str.casefold()
    out_eval = out_final.copy()
    out_eval["_join_name"] = out_eval["original_job_title"].astype(str).str.strip().str.casefold()
    joined = out_eval.merge(
        gt_eval[["_join_name", GT_COLS["major"], GT_COLS["minor"]]],
        on="_join_name", how="inner", suffixes=("_pred","_gt")
    )
    print("Overlap with Ground Truth:", len(joined))
    if len(joined):
        acc_major = (joined["major_role_group_pred"] == joined[GT_COLS["major"]]).mean()
        acc_minor = (joined["minor_sub_group_pred"] == joined[GT_COLS["minor"]]).mean()
        print(f"Accuracy — Major: {acc_major:.3f}, Minor: {acc_minor:.3f}")
else:
    print("GT columns not fully detected; skipping evaluation.")


# Save & Copy Results to Google Drive – saves the canonical file classified_job_descriptions.csv and copies it (plus inputs) into a timestamped folder in Drive.

In [ ]:
# Save final outputs
out_final.to_csv(OUTPUT_FINAL_CSV, index=False, encoding="utf-8")
out_final.to_csv(OUTPUT_CANONICAL, index=False, encoding="utf-8")
print("Saved ->", OUTPUT_FINAL_CSV, "and", OUTPUT_CANONICAL)

# Copy to Google Drive (works only in Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    base_target_folder = '/content/drive/My Drive/Colab Notebooks/Run Results'

    timestamp = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
    unique_folder_name = f'RUN_{timestamp}'
    target_folder = os.path.join(base_target_folder, unique_folder_name)
    os.makedirs(target_folder, exist_ok=True)

    files_to_copy = [
        '/content/Batch Input.csv',
        '/content/MNPS Roles.csv',
        '/content/MNPS KSACs.csv',
        '/content/Competency Extended Descriptions.csv',
        '/content/Korn_Ferry Lominger 38 Competencies.csv',
        '/content/Ground Truth Masterfile.csv',
        '/content/' + OUTPUT_CANONICAL
    ]

    for file_path in files_to_copy:
        try:
            file_name = os.path.basename(file_path)
            src = file_path if os.path.exists(file_path) else os.path.join('/content', file_name)
            destination_path = os.path.join(target_folder, file_name)
            shutil.copy(src, destination_path)
            print(f"Successfully copied {file_name} to {destination_path}")
        except FileNotFoundError:
            print(f"Error: {file_name} not found at {file_path}.")
        except Exception as e:
            print(f"Error copying file {file_name}: {e}")
except Exception as e:
    print("Google Drive copy step skipped (not running in Colab or Drive not available).", e)
